## Bureau Feature Engineering

In [1]:
import os
import sys

sys.path.append(os.path.abspath(".."))

In [2]:
import pandas as pd
import numpy as np
from src.data_utils import load_raw, save_interim

In [3]:
df_bureau=load_raw("bureau.csv")
df_bureau.head()

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


### 1. Base Aggregation (Loan Count)

In [4]:
bureau_loan_count=df_bureau.groupby('SK_ID_CURR').agg({
    'SK_ID_BUREAU':'count'
})

### 2. Credit Status Counts + Ratios

In [5]:
credit_status=df_bureau.groupby(
    ['SK_ID_CURR', 'CREDIT_ACTIVE'])['SK_ID_BUREAU'].count().unstack(fill_value=0)

credit_status.columns= [
    'BUREAU_' + col.upper().replace(' ','_') + '_COUNT' 
    for col in credit_status.columns]

#Add Ratios
total_loan=credit_status.sum(axis=1)

credit_status['BUREAU_ACTIVE_RATIO']=(
    credit_status.get('BUREAU_ACTIVE_COUNT', 0)/
    total_loan.replace(0,np.nan)
)

credit_status['BUREAU_BAD_DEBT_RATIO']=(
    credit_status.get('BUREAU_BAD_DEBT_COUNT', 0)/
    total_loan.replace(0,np.nan)
)

### 3. Credit Exposure & Utilization

In [6]:
credit_exposure=df_bureau.groupby('SK_ID_CURR').agg(
    BUREAU_TOTAL_CREDIT = ('AMT_CREDIT_SUM', 'sum',),
    BUREAU_TOTAL_DEBT = ('AMT_CREDIT_SUM_DEBT', 'sum')
)

credit_exposure['BUREAU_UTILIZATION_RATIO']=(
    credit_exposure['BUREAU_TOTAL_DEBT']/
    credit_exposure['BUREAU_TOTAL_CREDIT'].replace(0, np.nan)
)

### 4. Overdue Features

In [7]:
overdue_agg=df_bureau.groupby('SK_ID_CURR').agg(
    BUREAU_MAX_DAYS_OVERDUE = ('CREDIT_DAY_OVERDUE', 'max'),
    BUREAU_MEAN_DAYS_OVERDUE = ('CREDIT_DAY_OVERDUE', 'mean'),
    BUREAU_SUM_DAYS_OVERDUE = ('CREDIT_DAY_OVERDUE', 'sum'),
    BUREAU_MAX_OVERDUE_AMOUNT = ('AMT_CREDIT_SUM_OVERDUE', 'max'),
    BUREAU_MEAN_OVERDUE_AMOUNT = ('AMT_CREDIT_SUM_OVERDUE', 'mean'),
    BUREAU_TOTAL_OVERDUE_SUM = ('AMT_CREDIT_SUM_OVERDUE', 'sum')
)

### 5. Overdue Behavior Indicator

In [8]:
df_bureau['HAS_OVERDUE']=(df_bureau['CREDIT_DAY_OVERDUE']>0).astype(int)

overdue_flag=df_bureau.groupby('SK_ID_CURR').agg(
    BUREAU_OVERDUE_RATIO = ('HAS_OVERDUE', 'mean')
)